# Chat with Historical LLM (SFT Final)

Load the SFT-trained model and test its conversational abilities.

In [2]:
import sys
import os
sys.path.insert(0, r'C:\Users\danielyoon\Dropbox\hist_LLM\nanochat')

# Point to the model directory so the correct tokenizer (32768 vocab) is loaded
os.environ['NANOCHAT_BASE_DIR'] = r'D:\nanochat_model\1950_1999'

import torch
from nanochat.checkpoint_manager import build_model
from nanochat.engine import Engine

# Load model from custom checkpoint path
CHECKPOINT_DIR = r'D:\nanochat_model\1950_1999\sft_final'
STEP = 24269
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model, tokenizer, meta = build_model(CHECKPOINT_DIR, STEP, DEVICE, phase='eval')
engine = Engine(model, tokenizer)
print(f"Model loaded: {meta['model_config']}")
print(f"Device: {DEVICE}")
print(f"Step: {meta['step']}, Val loss: {meta['val_loss']:.4f}")
print(f"HellaSwag: {meta.get('hellaswag_acc', 'N/A')}, Winogrande: {meta.get('winogrande_acc', 'N/A')}")

2026-02-18 00:22:21,668 - nanochat.checkpoint_manager - INFO - Building model with config: {'sequence_len': 2048, 'vocab_size': 32768, 'n_layer': 23, 'n_head': 12, 'n_kv_head': 12, 'n_embd': 1536, 'window_pattern': 'L'}


Model loaded: {'sequence_len': 2048, 'vocab_size': 32768, 'n_layer': 23, 'n_head': 12, 'n_kv_head': 12, 'n_embd': 1536, 'window_pattern': 'L'}
Device: cpu
Step: 24269, Val loss: 1.2705
HellaSwag: 0.283203125, Winogrande: 0.4990234375


In [3]:
# Helper: chat with the model
def chat(prompt, temperature=0.8, top_k=50, max_tokens=256):
    """Send a single user message and get the model's response."""
    bos = tokenizer.get_bos_token_id()
    user_start = tokenizer.encode_special('<|user_start|>')
    user_end = tokenizer.encode_special('<|user_end|>')
    assistant_start = tokenizer.encode_special('<|assistant_start|>')
    assistant_end = tokenizer.encode_special('<|assistant_end|>')

    tokens = [bos, user_start]
    tokens.extend(tokenizer.encode(prompt))
    tokens.extend([user_end, assistant_start])

    response_tokens = []
    for token_col, token_mask in engine.generate(
        tokens, num_samples=1, max_tokens=max_tokens,
        temperature=temperature, top_k=top_k
    ):
        tok = int(token_col[0])
        if tok == assistant_end or tok == bos:
            break
        response_tokens.append(tok)

    return tokenizer.decode(response_tokens)


def multi_turn(messages, temperature=0.8, top_k=50, max_tokens=256):
    """Multi-turn conversation. messages = list of (role, content) tuples."""
    bos = tokenizer.get_bos_token_id()
    user_start = tokenizer.encode_special('<|user_start|>')
    user_end = tokenizer.encode_special('<|user_end|>')
    assistant_start = tokenizer.encode_special('<|assistant_start|>')
    assistant_end = tokenizer.encode_special('<|assistant_end|>')

    tokens = [bos]
    for role, content in messages:
        if role == 'user':
            tokens.append(user_start)
            tokens.extend(tokenizer.encode(content))
            tokens.append(user_end)
        elif role == 'assistant':
            tokens.append(assistant_start)
            tokens.extend(tokenizer.encode(content))
            tokens.append(assistant_end)
    tokens.append(assistant_start)

    response_tokens = []
    for token_col, token_mask in engine.generate(
        tokens, num_samples=1, max_tokens=max_tokens,
        temperature=temperature, top_k=top_k
    ):
        tok = int(token_col[0])
        if tok == assistant_end or tok == bos:
            break
        response_tokens.append(tok)

    return tokenizer.decode(response_tokens)

print('Chat helpers ready.')

Chat helpers ready.


## Test 1: Basic Conversation

In [4]:
for prompt in [
    "Hello, how are you?",
    "What is your name?",
    "Can you help me with a question?",
]:
    print(f"User: {prompt}")
    print(f"Model: {chat(prompt)}")
    print()

User: Hello, how are you?
Model: Hello, I'm a programming AI assistant, and I'm here to help you with any questions or problems you might have. What's on your mind?

User: What is your name?
Model: I am Amy Thorne. I'm a highly talented artist with a passion for capturing the beauty in the everyday.

User: Can you help me with a question?
Model: I'm sorry to hear about the delay in resolving your email, but as a fitness AI, I don't have the capability to assist with personal questions or emails. However, I'd be happy to help you find a fitness-related concern or guide. What's on your mind?



## Test 2: Historical Knowledge (Domain Data)

In [8]:
for prompt in [
    "What were the main causes of the Korean War?",
    "Describe the economic impact of the Marshall Plan on Western Europe.",
    "What was the significance of Brown v. Board of Education?",
    "Explain the Cuban Missile Crisis of 1962.",
]:
    print(f"User: {prompt}")
    print(f"Model: {chat(prompt, max_tokens=512)}")
    print()

User: What were the main causes of the Korean War?
Model: The Korean War was the result of the failed attempt by the Allies to contain the Communists after the invasion of China. The main causes of the Korean War include the failure of the Allies to achieve their goal of liberating the country from the Chinese's control.

Critics argued that the Allies had not handled the situation with the urgency and swiftness required for a peaceful resolution. They claimed that the Chinese were not prepared to make significant advances in nuclear technology and that theAllies had failed to show the necessary leadership to achieve their goals.

Another key factor was the failure of the Communists to negotiate effectively. The Allies did not present a united front, and many key players fell out of favor due to lack of communication and cooperation. However, the Allies ultimately managed to secure an early victory, and the Chinese were allowed to retreat.

Troubled leadership and poor strategic planni

## Test 3: Multiple Choice (Format the Model Was Trained On)

In [6]:
mc_questions = [
    """Multiple Choice question: What event marked the beginning of the Space Age?
- The launch of Sputnik 1 by the Soviet Union in 1957=A
- The first human-powered flight by the Wright Brothers in 1903=B
- The detonation of the first atomic bomb in 1945=C
- The founding of NASA in 1958=D

Respond only with the letter of the correct answer.""",

    """Multiple Choice question: Which treaty established the European Economic Community in 1957?
- Treaty of Versailles=A
- Treaty of Rome=B
- Treaty of Paris=C
- Treaty of Maastricht=D

Respond only with the letter of the correct answer.""",

    """Multiple Choice question: The Bretton Woods Agreement of 1944 established which of the following?
- The United Nations Security Council=A
- A system of fixed exchange rates pegged to the US dollar=B
- The North Atlantic Treaty Organization=C
- The General Agreement on Tariffs and Trade=D

Respond only with the letter of the correct answer.""",
]

correct = ['A', 'B', 'B']
for q, ans in zip(mc_questions, correct):
    response = chat(q, temperature=0.0, max_tokens=8)  # greedy, short
    response_clean = response.strip()[:1]  # first character
    match = 'Y' if response_clean == ans else 'N'
    print(f"Expected: {ans} | Model: {response.strip()!r} | Correct: {match}")

Expected: A | Model: 'B' | Correct: N
Expected: B | Model: 'B' | Correct: Y
Expected: B | Model: 'B' | Correct: Y


## Test 4: Reasoning / Explanation

In [7]:
for prompt in [
    "Why is the sky blue?",
    "What is 15 + 27?",
    "If it takes 5 machines 5 minutes to make 5 widgets, how long would it take 100 machines to make 100 widgets?",
]:
    print(f"User: {prompt}")
    print(f"Model: {chat(prompt, max_tokens=512)}")
    print()

User: Why is the sky blue?
Model: The sky appears blue when the Sun, Earth, and Mercury are aligned in a straight line.

User: What is 15 + 27?
Model: 15 + 27 is 40.

User: If it takes 5 machines 5 minutes to make 5 widgets, how long would it take 100 machines to make 100 widgets?
Model: If 5 machines make 5 widgets in 5 minutes, then each machine makes 1 widget in 5 minutes.
If we have 100 machines, it will still take them 5 minutes to make 100 widgets, since each machine is making 1 widget.
So, it would take 100 machines 5 minutes to make 100 widgets.
#### 5



## Test 5: Multi-Turn Conversation

In [9]:
# Test if the model can maintain context across turns
conversation = [
    ('user', 'What year did the Berlin Wall fall?'),
]
r1 = multi_turn(conversation, max_tokens=256)
print(f"User: {conversation[0][1]}")
print(f"Model: {r1}")
print()

conversation.append(('assistant', r1))
conversation.append(('user', 'What were the consequences of that event?'))
r2 = multi_turn(conversation, max_tokens=512)
print(f"User: What were the consequences of that event?")
print(f"Model: {r2}")
print()

conversation.append(('assistant', r2))
conversation.append(('user', 'Summarize what we discussed in one sentence.'))
r3 = multi_turn(conversation, max_tokens=256)
print(f"User: Summarize what we discussed in one sentence.")
print(f"Model: {r3}")

User: What year did the Berlin Wall fall?
Model: 1963

User: What were the consequences of that event?
Model: The falling of the Berlin Wall on August 17, 1963 led to the collapse of the East German government, which marked a turning point in the Cold War.

User: Summarize what we discussed in one sentence.
Model: We discussed the consequences of the Berlin Wall falling, focusing on its impact on the East German government and the broader Cold War context.


## Test 6: Temperature Comparison
Same prompt at different temperatures to see diversity vs coherence.

In [10]:
prompt = "Describe the Cold War in three sentences."
for temp in [0.0, 0.5, 0.8, 1.2]:
    response = chat(prompt, temperature=temp, max_tokens=256)
    print(f"Temperature {temp}: {response}")
    print()

Temperature 0.0: The Cold War was a period of tension between the United States and the Soviet Union, marked by the escalation of military tensions, economic rivalries, and the growing sense of nationalism in both superpowers.

Temperature 0.5: The Cold War was a period of heightened tensions between the United States and the Soviet Union, fueled by the ideological rivalry between communism and communism, as well as the ongoing global conflict over issues such as the Vietnam War and the Strategic Arms Limitation Talks.

Temperature 0.8: The Cold War was a period of heightened tensions between the United States and the Soviet Union, fueled by a desire to control the global economy and a lack of trust between the two nations. It began in the mid-1930s when the Nazis took advantage of the Soviet Union's advanced technology and the U.S. military, setting the stage for a series of events that would have far-reaching consequences.

Temperature 1.2: The Cold War was an era in which the US tro

## RACE Reading Comprehension Evaluation

RACE is a reading comprehension benchmark from English exams. The model receives a passage
and answers 4-choice MC questions about it. This tests reasoning from provided context —
no memorized facts needed. Random baseline = 25%.

| Level | Description | Our Model |
|-------|-------------|-----------|
| RACE-Middle | Middle school reading | **58%** |
| RACE-High | High school reading | **40%** |
| Combined | | **49%** |
| Random | | 25% |

In [17]:
from datasets import load_dataset

letters = ['A', 'B', 'C', 'D']

def eval_race(subset, n=100):
    """Evaluate on RACE reading comprehension (4-choice MC with passage)."""
    ds = load_dataset("ehovy/race", subset, split="test")
    correct = 0
    total = 0
    skipped = 0
    for i in range(min(n, len(ds))):
        ex = ds[i]
        prompt = f"Read the following passage and answer the question.\n\n"
        prompt += f"Passage: {ex['article']}\n\n"
        prompt += f"Multiple Choice question: {ex['question']}\n"
        for letter, choice in zip(letters, ex['options']):
            prompt += f"- {choice}={letter}\n"
        prompt += "\nRespond only with the letter of the correct answer."

        response = chat(prompt, temperature=0.0, max_tokens=8)
        if response == "[TOO LONG]":
            skipped += 1
            continue

        print(prompt)

        pred = response.strip()[:1].upper()
        is_correct = pred == ex['answer']
        if is_correct:
            correct += 1
        total += 1

        mark = "OK" if is_correct else "XX"
        if i < 10:  # show first 10 details
            print(f"  [{mark}] Q{i+1}: expected={ex['answer']}, got={pred!r} | {ex['question'][:60]}...")

    acc = correct / total if total > 0 else 0
    print(f"\n  {subset}: {correct}/{total} = {acc:.1%} (skipped {skipped} long passages)\n")
    return correct, total

print("=" * 60)
print("RACE Reading Comprehension (100 per subset)")
print("=" * 60)
print("\nRACE-Middle:")
m_c, m_t = eval_race("middle", 100)
print("RACE-High:")
h_c, h_t = eval_race("high", 100)

print("=" * 60)
print(f"  RACE-Middle: {m_c}/{m_t} = {m_c/m_t:.1%}")
print(f"  RACE-High:   {h_c}/{h_t} = {h_c/h_t:.1%}")
print(f"  Combined:    {m_c+h_c}/{m_t+h_t} = {(m_c+h_c)/(m_t+h_t):.1%}")
print(f"  Random:      25.0%")
print("=" * 60)

RACE Reading Comprehension (100 per subset)

RACE-Middle:
Read the following passage and answer the question.

Passage: It is well-known that the "prom", a formal dance held at the end of high school or college, is an important date in every student's life. What is less well-known is that the word "prom" comes from the verb "to promenade", which means to walk around, beautifully dressed, in order to attract attention. The idea is that you should see and be seen by others.
The prom is not just an American tradition, though most people believe that it started in America. In Canada the event is called a "formal". In Britain and Australia the old fashioned word "dance" is more and more frequently being referred to as a "prom". Most countries have some form of celebration when students finish high school: after all, it means the end of life as a child, and the beginning of life as an adult.
The prom is expensive to organize and the tickets can cost students a lot of money. The tradition is 

## Interactive Chat
Run this cell to chat interactively. Type 'quit' to stop.

## Comprehensive Benchmark Evaluation

Tested 10 benchmarks (50 examples each, greedy decoding) to map the model's strengths and weaknesses.

### Results Summary

| Benchmark | Accuracy | Choices | Random | vs Random | Prediction Bias | Verdict |
|-----------|----------|---------|--------|-----------|-----------------|---------|
| **RACE-Middle** | **62%** | 4 | 25% | **+37pp** | Diverse (A:5 B:19 C:15 D:11) | **GENUINE SKILL** |
| PIQA | 58% | 2 | 50% | +8pp | B:48/50 = always B | Inflated by B-bias |
| BoolQ | 58% | 2 | 50% | +8pp | B:50/50 = always B | Entirely B-bias |
| Winogrande | 46% | 2 | 50% | -4pp | B:48/50 = always B | At random (biased) |
| **RACE-High** | **38%** | 4 | 25% | **+13pp** | Diverse (A:3 B:11 C:14 D:22) | **GENUINE SKILL** |
| **ARC-Easy** | **34%** | 4 | 25% | **+9pp** | B-heavy but diverse | Marginal signal |
| **HellaSwag** | **32%** | 4 | 25% | **+7pp** | B:19 C:14 D:17 | Marginal signal |
| MMLU (test) | 28% | 4 | 25% | +3pp | B:18 C:17 D:14 | At random |
| CommonsenseQA | 28% | 5 | 20% | +8pp | E:45/50 = always E | Entirely E-bias |
| ARC-Challenge | 20% | 4 | 25% | -5pp | B:20 C:13 D:16 | Below random |

### Key Findings

**1. Reading comprehension is the model's genuine strength.**
RACE-Middle (62%) and RACE-High (38%) have diverse prediction distributions matching the answer 
distributions, confirming the model is actually reading passages and selecting answers based on content.

**2. The model has a severe B-bias on 2-choice tasks.**
For Winogrande, PIQA, and BoolQ, the model picks B 96-100% of the time. PIQA's apparent 58% is 
an artifact: the dataset happens to have 58% B answers. BoolQ (always picks "no") similarly.
These benchmarks are uninformative for this model.

**3. Knowledge-dependent benchmarks (MMLU, ARC-Challenge) are at or below random.**
Without a passage to reason from, the model cannot answer knowledge questions. This is expected 
for a ~200M model trained on domain-specific historical data.

**4. CommonsenseQA has an E-bias (always picks last option).**
The model picks E (the 5th option) 90% of the time on CommonsenseQA, a positional bias.

### Recommendations for Eval Suite

| Add to eval? | Benchmark | Why |
|:---:|-----------|-----|
| **YES** | RACE-Middle | Genuine signal, tests reasoning from context, well above random |
| **YES** | RACE-High | Genuine signal, harder version provides ceiling measure |
| Keep | HellaSwag | Already in eval, marginal but real signal on 4-choice |
| Drop | Winogrande | B-bias makes it uninformative (always ~50%) |
| Drop | PIQA | Same B-bias problem |
| Maybe | ARC-Easy | Weak signal (+9pp), could track improvement over time |
| No | BoolQ | Always predicts B, uninformative |
| No | MMLU test | At random without passage context |
| No | CommonsenseQA | E-bias, uninformative |
| No | ARC-Challenge | Below random |

In [ ]:
conversation = []
print("Chat with the Historical LLM. Type 'quit' to exit, 'reset' to clear history.")
print()
while True:
    user_input = input("You: ").strip()
    if user_input.lower() == 'quit':
        break
    if user_input.lower() == 'reset':
        conversation = []
        print("[Conversation reset]\n")
        continue
    if not user_input:
        continue
    conversation.append(('user', user_input))
    response = multi_turn(conversation, max_tokens=512)
    conversation.append(('assistant', response))
    print(f"Model: {response}\n")

In [ ]:
# Run comprehensive evaluation across all benchmarks (takes ~20 min on CPU)
import json
from collections import Counter

sys.path.insert(0, r'C:\Users\danielyoon\Dropbox\hist_LLM\nanochat')
from tasks.mmlu import MMLU
from tasks.arc import ARC
from tasks.hellaswag import HellaSwag
from tasks.winogrande import Winogrande
from tasks.piqa import PIQA
from tasks.boolq import BoolQ
from tasks.customjson import CustomJSON
from datasets import load_dataset

N = 50  # examples per benchmark

def eval_task(name, dataset, n=N):
    """Evaluate a nanochat-format task (messages with user prompt + expected answer)."""
    correct, total, skipped = 0, 0, 0
    pred_dist, answer_dist = Counter(), Counter()
    samples = []
    for idx in range(min(n, len(dataset))):
        ex = dataset[idx]
        prompt = ex['messages'][0]['content']
        expected = ex['messages'][1]['content'].strip()
        response = chat(prompt, temperature=0.0, max_tokens=8)
        if response == "[TOO LONG]":
            skipped += 1; continue
        pred = response.strip()[:1].upper()
        expected_l = expected[:1].upper()
        pred_dist[pred] += 1; answer_dist[expected_l] += 1
        is_correct = pred == expected_l
        if is_correct: correct += 1
        total += 1
        if len(samples) < 3:
            samples.append({'q': prompt[:150], 'expected': expected, 'predicted': pred, 'ok': is_correct})
    acc = correct / total if total > 0 else 0
    return {'name': name, 'acc': acc, 'correct': correct, 'total': total,
            'pred_dist': dict(sorted(pred_dist.items())),
            'answer_dist': dict(sorted(answer_dist.items())), 'samples': samples}

def eval_race_subset(subset, n=N):
    """Evaluate RACE reading comprehension."""
    ds = load_dataset("ehovy/race", subset, split="test")
    correct, total = 0, 0
    pred_dist, answer_dist = Counter(), Counter()
    samples = []
    mc_letters = ['A', 'B', 'C', 'D']
    for i in range(min(n, len(ds))):
        ex = ds[i]
        prompt = f"Read the following passage and answer the question.\n\nPassage: {ex['article']}\n\n"
        prompt += f"Multiple Choice question: {ex['question']}\n"
        for letter, choice in zip(mc_letters, ex['options']):
            prompt += f"- {choice}={letter}\n"
        prompt += "\nRespond only with the letter of the correct answer."
        response = chat(prompt, temperature=0.0, max_tokens=8)
        if response == "[TOO LONG]": continue
        pred = response.strip()[:1].upper()
        pred_dist[pred] += 1; answer_dist[ex['answer']] += 1
        is_correct = pred == ex['answer']
        if is_correct: correct += 1
        total += 1
        if len(samples) < 3:
            samples.append({'q': ex['question'][:100], 'expected': ex['answer'], 'predicted': pred, 'ok': is_correct})
    acc = correct / total if total > 0 else 0
    return {'name': f"RACE-{subset.capitalize()}", 'acc': acc, 'correct': correct, 'total': total,
            'pred_dist': dict(sorted(pred_dist.items())),
            'answer_dist': dict(sorted(answer_dist.items())), 'samples': samples}

# Run all benchmarks
benchmarks = [
    ("MMLU (test)", lambda: MMLU(subset="all", split="test", stop=N)),
    ("ARC-Easy", lambda: ARC(subset="ARC-Easy", split="test", stop=N)),
    ("ARC-Challenge", lambda: ARC(subset="ARC-Challenge", split="test", stop=N)),
    ("HellaSwag", lambda: HellaSwag(split="validation", stop=N)),
    ("Winogrande", lambda: Winogrande(split="validation", stop=N)),
    ("PIQA", lambda: PIQA(split="validation", stop=N)),
    ("BoolQ", lambda: BoolQ(split="validation", stop=N)),
    ("CommonsenseQA", lambda: CustomJSON(
        filepath=r'C:\Users\danielyoon\Dropbox\hist_LLM\data\periods_data\1950_1999\posttraining_data\final\test\commonsenseqa_test.jsonl',
        stop=N)),
]

results = []
for name, loader in benchmarks:
    print(f"Evaluating {name}...", end=" ", flush=True)
    r = eval_task(name, loader(), n=N)
    results.append(r)
    print(f"{r['acc']:.0%} ({r['correct']}/{r['total']})  preds={r['pred_dist']}")

for subset in ["middle", "high"]:
    print(f"Evaluating RACE-{subset.capitalize()}...", end=" ", flush=True)
    r = eval_race_subset(subset, n=N)
    results.append(r)
    print(f"{r['acc']:.0%} ({r['correct']}/{r['total']})  preds={r['pred_dist']}")

# Print sample predictions for each
print("\n" + "=" * 70)
print("SAMPLE PREDICTIONS")
print("=" * 70)
for r in sorted(results, key=lambda x: x['acc'], reverse=True):
    print(f"\n{r['name']} -- {r['acc']:.0%}  (answers={r['answer_dist']}, preds={r['pred_dist']})")
    for s in r['samples']:
        mark = "OK" if s['ok'] else "XX"
        print(f"  [{mark}] expected={s['expected']}, got={s['predicted']}  Q: {s['q'][:80]}...")